# Benchmarking CVRPTW: All Gehring & Homberger 200-Customer Instances with cuOpt

The **Gehring & Homberger** benchmark suite is one of the most widely used evaluation frameworks for the **Capacitated Vehicle Routing Problem with Time Windows (CVRPTW)**. It covers 200, 400, 600, 800, and 1000 customer problem sizes, with six distribution families per size.

This notebook focuses on all **60 instances** in the **200-customer** set and systematically tests each one using the **Vehicle Route Optimizer** from the **OCI AI Accelerator Pack** (powered by **NVIDIA cuOpt**). Results are collected for every instance and summarised in a comparison table so you can see at a glance how cuOpt performs across the full range of problem characteristics.

**Relationship to `intro.ipynb`**: The companion notebook `intro.ipynb` provides a detailed, step-by-step walkthrough of CVRPTW concepts (data structures, payload building, time-window analysis) using two representative instances (`C1_2_1` and `R1_2_1`). This notebook assumes that background knowledge and focuses instead on breadth — running all 60 instances and comparing outcomes.

## Table of Contents
1. [Install Dependencies](#1-install-dependencies)
2. [Set Up Helper Functions](#2-set-up-helper-functions)
3. [Configure API Endpoint](#3-configure-api-endpoint)
4. [Dataset Overview](#4-dataset-overview)
5. [Define All Benchmark Instances](#5-define-all-benchmark-instances)
6. [Download Benchmark Data](#6-download-benchmark-data)
7. [Run Optimisation for All Instances](#7-run-optimisation-for-all-instances)
8. [Results Summary Table](#8-results-summary-table)
9. [Conclusion](#9-conclusion)
10. [Cleanup](#10-cleanup)

## 1. Install Dependencies

In [ ]:
# The following packages are required to run this notebook:
%pip install -q matplotlib scipy pandas requests

import os
import sys
import requests
import logging
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO)

## 2. Set Up Helper Functions

All utility functions are imported from `helper/utils.py` — the same module used by `intro.ipynb`.
See that notebook for a full description of each helper.

In [ ]:
# Add the helper directory to the Python module search path
sys.path.insert(0, './helper')

from utils import (
    plot_routes,
    plot_instance,
    create_from_file,
    build_payload,
    solve,
)

## 3. Configure API Endpoint

Set `BASE_URL` to the endpoint of your deployed Vehicle Route Optimizer. You can find this URL in the
**Outputs** tab of the OCI Resource Manager stack after deployment completes.

In [ ]:
# Base URL of the cuOpt optimisation server
# Update this value to point to your own deployed cuOpt instance
base_url = "http://<Replace with your cuOpt Endpoint>"


In [ ]:
# Verify that the cuOpt server is reachable before running the benchmark loop
health_url = f"{base_url}/cuopt/health"
try:
    response = requests.get(health_url)
    print(f"Health check URL : {health_url}")
    print(f"Status code      : {response.status_code}")
    print(f"Response         : {response.json()}")
except requests.RequestException as e:
    print(f"Health check failed: {e}")

## 4. Dataset Overview

The **Gehring & Homberger 200-customer benchmark** consists of 60 problem instances, organised into
six families of ten instances each:

| Family | Prefix | Customer Distribution | Time-Window Type |
|--------|--------|-----------------------|------------------|
| C1 | `C1_2_*` | Clustered | Narrow |
| C2 | `C2_2_*` | Clustered | Wide |
| R1 | `R1_2_*` | Random | Narrow |
| R2 | `R2_2_*` | Random | Wide |
| RC1 | `RC1_2_*` | Mixed (Random + Clustered) | Narrow |
| RC2 | `RC2_2_*` | Mixed (Random + Clustered) | Wide |

All instances have **200 customers** and **1 depot**. Each family tests a different combination of
spatial structure and time-window tightness, creating a diverse and challenging evaluation suite.

**Naming convention**: `X_2_N` where `X` is the family prefix, `2` denotes 200-customer scale, and
`N` ∈ {1, …, 10} is the instance number within the family.

### Solver Configuration
Each instance is solved with a single, configurable `TIME_LIMIT` (seconds). Increase this value to
allow cuOpt more time to refine routes at the cost of a longer total run time.

## 5. Define All Benchmark Instances

In [ ]:
# ── Solver time limit (seconds per instance) ────────────────────────────
# Increase for better solution quality; decrease for faster iteration.
TIME_LIMIT = 60

# ── Best-known solutions for all 60 instances ────────────────────────────
# Source: SINTEF TOP — https://www.sintef.no/top/vrptw/homberger-benchmark/
# Objective: minimise (1) number of vehicles, then (2) total travel cost.
best_known = {
    # ── C1: Clustered, narrow time windows ──────────────────────────────
    "C1_2_1" : {"n_vehicles": 20, "cost": 2704.57, "family": "C1", "distribution": "Clustered", "tw_type": "Narrow"},
    "C1_2_2" : {"n_vehicles": 18, "cost": 2917.89, "family": "C1", "distribution": "Clustered", "tw_type": "Narrow"},
    "C1_2_3" : {"n_vehicles": 18, "cost": 2707.35, "family": "C1", "distribution": "Clustered", "tw_type": "Narrow"},
    "C1_2_4" : {"n_vehicles": 18, "cost": 2643.31, "family": "C1", "distribution": "Clustered", "tw_type": "Narrow"},
    "C1_2_5" : {"n_vehicles": 20, "cost": 2702.05, "family": "C1", "distribution": "Clustered", "tw_type": "Narrow"},
    "C1_2_6" : {"n_vehicles": 20, "cost": 2701.04, "family": "C1", "distribution": "Clustered", "tw_type": "Narrow"},
    "C1_2_7" : {"n_vehicles": 20, "cost": 2701.04, "family": "C1", "distribution": "Clustered", "tw_type": "Narrow"},
    "C1_2_8" : {"n_vehicles": 19, "cost": 2775.48, "family": "C1", "distribution": "Clustered", "tw_type": "Narrow"},
    "C1_2_9" : {"n_vehicles": 18, "cost": 2687.83, "family": "C1", "distribution": "Clustered", "tw_type": "Narrow"},
    "C1_2_10": {"n_vehicles": 18, "cost": 2643.51, "family": "C1", "distribution": "Clustered", "tw_type": "Narrow"},
    # ── C2: Clustered, wide time windows ────────────────────────────────
    "C2_2_1" : {"n_vehicles":  6, "cost": 1931.44, "family": "C2", "distribution": "Clustered", "tw_type": "Wide"},
    "C2_2_2" : {"n_vehicles":  6, "cost": 1863.16, "family": "C2", "distribution": "Clustered", "tw_type": "Wide"},
    "C2_2_3" : {"n_vehicles":  6, "cost": 1775.08, "family": "C2", "distribution": "Clustered", "tw_type": "Wide"},
    "C2_2_4" : {"n_vehicles":  6, "cost": 1703.43, "family": "C2", "distribution": "Clustered", "tw_type": "Wide"},
    "C2_2_5" : {"n_vehicles":  6, "cost": 1878.85, "family": "C2", "distribution": "Clustered", "tw_type": "Wide"},
    "C2_2_6" : {"n_vehicles":  6, "cost": 1857.35, "family": "C2", "distribution": "Clustered", "tw_type": "Wide"},
    "C2_2_7" : {"n_vehicles":  6, "cost": 1849.46, "family": "C2", "distribution": "Clustered", "tw_type": "Wide"},
    "C2_2_8" : {"n_vehicles":  6, "cost": 1820.53, "family": "C2", "distribution": "Clustered", "tw_type": "Wide"},
    "C2_2_9" : {"n_vehicles":  6, "cost": 1830.05, "family": "C2", "distribution": "Clustered", "tw_type": "Wide"},
    "C2_2_10": {"n_vehicles":  6, "cost": 1806.58, "family": "C2", "distribution": "Clustered", "tw_type": "Wide"},
    # ── R1: Random, narrow time windows ─────────────────────────────────
    "R1_2_1" : {"n_vehicles": 20, "cost": 4784.11, "family": "R1", "distribution": "Random",    "tw_type": "Narrow"},
    "R1_2_2" : {"n_vehicles": 18, "cost": 4039.86, "family": "R1", "distribution": "Random",    "tw_type": "Narrow"},
    "R1_2_3" : {"n_vehicles": 18, "cost": 3381.96, "family": "R1", "distribution": "Random",    "tw_type": "Narrow"},
    "R1_2_4" : {"n_vehicles": 18, "cost": 3057.81, "family": "R1", "distribution": "Random",    "tw_type": "Narrow"},
    "R1_2_5" : {"n_vehicles": 18, "cost": 4107.86, "family": "R1", "distribution": "Random",    "tw_type": "Narrow"},
    "R1_2_6" : {"n_vehicles": 18, "cost": 3583.14, "family": "R1", "distribution": "Random",    "tw_type": "Narrow"},
    "R1_2_7" : {"n_vehicles": 18, "cost": 3150.11, "family": "R1", "distribution": "Random",    "tw_type": "Narrow"},
    "R1_2_8" : {"n_vehicles": 18, "cost": 2951.99, "family": "R1", "distribution": "Random",    "tw_type": "Narrow"},
    "R1_2_9" : {"n_vehicles": 18, "cost": 3760.58, "family": "R1", "distribution": "Random",    "tw_type": "Narrow"},
    "R1_2_10": {"n_vehicles": 18, "cost": 3301.18, "family": "R1", "distribution": "Random",    "tw_type": "Narrow"},
    # ── R2: Random, wide time windows ───────────────────────────────────
    "R2_2_1" : {"n_vehicles":  4, "cost": 4483.16, "family": "R2", "distribution": "Random",    "tw_type": "Wide"},
    "R2_2_2" : {"n_vehicles":  4, "cost": 3621.20, "family": "R2", "distribution": "Random",    "tw_type": "Wide"},
    "R2_2_3" : {"n_vehicles":  4, "cost": 2880.62, "family": "R2", "distribution": "Random",    "tw_type": "Wide"},
    "R2_2_4" : {"n_vehicles":  4, "cost": 1981.29, "family": "R2", "distribution": "Random",    "tw_type": "Wide"},
    "R2_2_5" : {"n_vehicles":  4, "cost": 3366.79, "family": "R2", "distribution": "Random",    "tw_type": "Wide"},
    "R2_2_6" : {"n_vehicles":  4, "cost": 2913.03, "family": "R2", "distribution": "Random",    "tw_type": "Wide"},
    "R2_2_7" : {"n_vehicles":  4, "cost": 2451.14, "family": "R2", "distribution": "Random",    "tw_type": "Wide"},
    "R2_2_8" : {"n_vehicles":  4, "cost": 1849.87, "family": "R2", "distribution": "Random",    "tw_type": "Wide"},
    "R2_2_9" : {"n_vehicles":  4, "cost": 3092.04, "family": "R2", "distribution": "Random",    "tw_type": "Wide"},
    "R2_2_10": {"n_vehicles":  4, "cost": 2654.97, "family": "R2", "distribution": "Random",    "tw_type": "Wide"},
    # ── RC1: Mixed random+clustered, narrow time windows ────────────────
    "RC1_2_1" : {"n_vehicles": 18, "cost": 3602.80, "family": "RC1", "distribution": "Mixed",     "tw_type": "Narrow"},
    "RC1_2_2" : {"n_vehicles": 18, "cost": 3249.05, "family": "RC1", "distribution": "Mixed",     "tw_type": "Narrow"},
    "RC1_2_3" : {"n_vehicles": 18, "cost": 3008.33, "family": "RC1", "distribution": "Mixed",     "tw_type": "Narrow"},
    "RC1_2_4" : {"n_vehicles": 18, "cost": 2851.68, "family": "RC1", "distribution": "Mixed",     "tw_type": "Narrow"},
    "RC1_2_5" : {"n_vehicles": 18, "cost": 3371.00, "family": "RC1", "distribution": "Mixed",     "tw_type": "Narrow"},
    "RC1_2_6" : {"n_vehicles": 18, "cost": 3324.80, "family": "RC1", "distribution": "Mixed",     "tw_type": "Narrow"},
    "RC1_2_7" : {"n_vehicles": 18, "cost": 3189.32, "family": "RC1", "distribution": "Mixed",     "tw_type": "Narrow"},
    "RC1_2_8" : {"n_vehicles": 18, "cost": 3083.93, "family": "RC1", "distribution": "Mixed",     "tw_type": "Narrow"},
    "RC1_2_9" : {"n_vehicles": 18, "cost": 3081.13, "family": "RC1", "distribution": "Mixed",     "tw_type": "Narrow"},
    "RC1_2_10": {"n_vehicles": 18, "cost": 3000.30, "family": "RC1", "distribution": "Mixed",     "tw_type": "Narrow"},
    # ── RC2: Mixed random+clustered, wide time windows ───────────────────
    "RC2_2_1" : {"n_vehicles":  6, "cost": 3099.53, "family": "RC2", "distribution": "Mixed",     "tw_type": "Wide"},
    "RC2_2_2" : {"n_vehicles":  5, "cost": 2825.24, "family": "RC2", "distribution": "Mixed",     "tw_type": "Wide"},
    "RC2_2_3" : {"n_vehicles":  4, "cost": 2601.87, "family": "RC2", "distribution": "Mixed",     "tw_type": "Wide"},
    "RC2_2_4" : {"n_vehicles":  4, "cost": 2038.56, "family": "RC2", "distribution": "Mixed",     "tw_type": "Wide"},
    "RC2_2_5" : {"n_vehicles":  4, "cost": 2911.46, "family": "RC2", "distribution": "Mixed",     "tw_type": "Wide"},
    "RC2_2_6" : {"n_vehicles":  4, "cost": 2873.12, "family": "RC2", "distribution": "Mixed",     "tw_type": "Wide"},
    "RC2_2_7" : {"n_vehicles":  4, "cost": 2525.83, "family": "RC2", "distribution": "Mixed",     "tw_type": "Wide"},
    "RC2_2_8" : {"n_vehicles":  4, "cost": 2292.53, "family": "RC2", "distribution": "Mixed",     "tw_type": "Wide"},
    "RC2_2_9" : {"n_vehicles":  4, "cost": 2175.04, "family": "RC2", "distribution": "Mixed",     "tw_type": "Wide"},
    "RC2_2_10": {"n_vehicles":  4, "cost": 2015.60, "family": "RC2", "distribution": "Mixed",     "tw_type": "Wide"},
}

# Ordered list of instance names (C1 → C2 → R1 → R2 → RC1 → RC2)
instance_names = sorted(best_known.keys())

print(f"Total instances defined : {len(best_known)}")
print("Families:", sorted({v['family'] for v in best_known.values()}))

## 6. Download Benchmark Data

Download and extract the 200-customer Gehring & Homberger instance archive from the SINTEF TOP repository.
The ZIP file contains all 60 `.TXT` instance files. If you have already downloaded it, this cell will
skip the download and re-use the cached files.

In [ ]:
DATA_DIR = "./cvrptw_data"
ZIP_URL  = (
    "https://www.sintef.no/globalassets/project/top/vrptw/homberger/200/"
    "homberger_200_customer_instances.zip"
)
ZIP_PATH = os.path.join(DATA_DIR, "homberger_data.zip")

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    print("Downloading benchmark archive …")
    !wget -q {ZIP_URL} -O {ZIP_PATH}
    print("Download complete.")
else:
    print(f"Archive already present: {ZIP_PATH}")

!unzip -q -o {ZIP_PATH} -d {DATA_DIR}

# Verify that every expected instance file is present
missing = []
for name in instance_names:
    fp = os.path.join(DATA_DIR, f"{name}.TXT")
    if not os.path.exists(fp):
        missing.append(name)

if missing:
    print(f"WARNING — {len(missing)} file(s) not found: {missing}")
else:
    print(f"All {len(instance_names)} instance files found in {DATA_DIR}/")

## 7. Run Optimisation for All Instances

The cell below iterates over all 60 instances in the order **C1 → C2 → R1 → R2 → RC1 → RC2**.
For each instance it:

1. Loads the benchmark file into a DataFrame.
2. Visualises the customer locations and time-window distribution (`plot_instance`).
3. Builds the cuOpt payload (cost matrix + fleet data + task data).
4. Submits the problem to the solver with the configured `TIME_LIMIT`.
5. Visualises the returned routes (`plot_routes`).
6. Records the result (vehicles used, total cost, cost gap) into `all_results`.

> **Estimated run time**: ~`60 × TIME_LIMIT` seconds plus network/processing overhead. 

While this is in progress, you can monitor GPU performance via a Grafana dashboard powered by the NVIDIA DCGM exporter. The dashboard URL and login credentials are available in the Terraform output.

This setup is automatically installed as part of the AI Accelerator pack, so no additional configuration is required. Grafana is a visualization platform that lets you explore real-time and historical metrics through interactive dashboards, while the NVIDIA DCGM (Data Center GPU Manager) exporter collects detailed GPU telemetry such as utilization, memory usage, temperature, and power consumption. Together, they provide clear visibility into GPU performance, help quickly identify bottlenecks or failures, and enable more efficient optimization of workloads and resource usage.

![Grafana-Dashboard-1](./img/grafana-1.png)

![Grafana-Dashboard-2](./img/grafana-2.png)

You can also explore other dashboards, such as the All Pod Memory Usage dashboard, to monitor cluster resource consumption at a glance.

![Pod Memory](./img/pod-memory.png)

In [ ]:
# Results accumulator — keyed by instance name
all_results = {}

for instance_name in instance_names:
    bk    = best_known[instance_name]
    label = f"{bk['distribution']} / {bk['tw_type']} TW"
    fpath = os.path.join(DATA_DIR, f"{instance_name}.TXT")

    print("\n" + "=" * 72)
    print(f"Instance : {instance_name}   [{label}]")
    print(f"Best-known: {bk['n_vehicles']} vehicles, cost {bk['cost']:.2f}")
    print("=" * 72)

    # ── 1. Load instance ────────────────────────────────────────────────
    if not os.path.exists(fpath):
        print(f"  SKIP — file not found: {fpath}")
        all_results[instance_name] = {"error": "File not found"}
        continue

    orders, vehicle_cap, n_vehicles = create_from_file(fpath)
    n_locations = orders['demand'].shape[0] - 1
    print(f"  Customers: {n_locations}  |  Vehicles: {n_vehicles}  |  Capacity: {vehicle_cap}")

    # ── 2. Visualise instance ────────────────────────────────────────────
    plot_instance(orders, instance_name, label)

    # ── 3. Build payload ─────────────────────────────────────────────────
    payload = build_payload(orders, n_locations, n_vehicles, vehicle_cap)

    # ── 4. Solve ──────────────────────────────────────────────────────────
    solution = solve(base_url, payload, TIME_LIMIT)

    # ── 5. Visualise routes ───────────────────────────────────────────────
    if solution.get('status') == 0:
        n_veh_used = solution.get('num_vehicles')
        total_cost = solution.get('solution_cost')

        veh_gap  = n_veh_used - bk['n_vehicles']
        cost_pct = (total_cost / bk['cost'] - 1) * 100

        print(f"  cuOpt result : {n_veh_used} vehicles, cost {total_cost:.2f}")
        print(f"  Vehicle gap  : {veh_gap:+d}  |  Cost gap: {cost_pct:+.2f}%")

        plot_routes(
            solution,
            orders,
            title=f"{instance_name} ({label}) — {TIME_LIMIT}s limit",
            max_routes=20,
            detail_routes=5,
        )

        all_results[instance_name] = {
            "family"        : bk['family'],
            "distribution"  : bk['distribution'],
            "tw_type"       : bk['tw_type'],
            "num_vehicles"  : n_veh_used,
            "solution_cost" : total_cost,
            "bk_vehicles"   : bk['n_vehicles'],
            "bk_cost"       : bk['cost'],
            "vehicle_gap"   : veh_gap,
            "cost_gap_pct"  : cost_pct,
        }
    else:
        status = solution.get('status', 'N/A')
        print(f"  Optimisation did not return a feasible solution (status={status})")
        all_results[instance_name] = {
            "family"        : bk['family'],
            "distribution"  : bk['distribution'],
            "tw_type"       : bk['tw_type'],
            "num_vehicles"  : None,
            "solution_cost" : None,
            "bk_vehicles"   : bk['n_vehicles'],
            "bk_cost"       : bk['cost'],
            "vehicle_gap"   : None,
            "cost_gap_pct"  : None,
        }

print("\n" + "=" * 72)
print(f"Benchmark loop complete.  Instances processed: {len(all_results)}")
print("=" * 72)

## 8. Results Summary Table

The table below collects every instance result into a single view, making it easy to compare
cuOpt's performance across problem families and to identify where the solver excels or struggles.

In [ ]:
# Build the summary DataFrame
summary_rows = []
for name, r in all_results.items():
    if "error" in r:
        summary_rows.append({
            "Instance"          : name,
            "Family"            : r.get('family', '?'),
            "Distribution"      : r.get('distribution', '?'),
            "TW Type"           : r.get('tw_type', '?'),
            "cuOpt Vehicles"     : 'N/A',
            "BK Vehicles"        : r.get('bk_vehicles', 'N/A'),
            "Vehicle Gap"        : 'N/A',
            "cuOpt Cost"         : 'N/A',
            "BK Cost"            : 'N/A',
            "Cost Gap (%)"       : 'N/A',
        })
        continue

    n_veh   = r['num_vehicles']
    cost    = r['solution_cost']
    bk_veh  = r['bk_vehicles']
    bk_cost = r['bk_cost']

    summary_rows.append({
        "Instance"          : name,
        "Family"            : r['family'],
        "Distribution"      : r['distribution'],
        "TW Type"           : r['tw_type'],
        "cuOpt Vehicles"     : n_veh    if n_veh  is not None else 'N/A',
        "BK Vehicles"        : bk_veh,
        "Vehicle Gap"        : (f"{r['vehicle_gap']:+d}"
                                 if r['vehicle_gap'] is not None else 'N/A'),
        "cuOpt Cost"         : (f"{cost:.2f}"
                                 if cost is not None else 'N/A'),
        "BK Cost"            : f"{bk_cost:.2f}",
        "Cost Gap (%)"       : (f"{r['cost_gap_pct']:+.2f}%"
                                 if r['cost_gap_pct'] is not None else 'N/A'),
    })

summary_df = pd.DataFrame(summary_rows).set_index('Instance')

print("=" * 80)
print(f"cuOpt Performance Summary — All 200-Customer Gehring & Homberger Instances")
print(f"Solver time limit: {TIME_LIMIT} s per instance")
print("=" * 80)

try:
    from IPython.display import display
    display(summary_df)
except ImportError:
    print(summary_df.to_string())

In [ ]:
# Per-family aggregate statistics
numeric_df = summary_df[
    summary_df['cuOpt Vehicles'] != 'N/A'
].copy()
numeric_df['cuOpt Vehicles'] = pd.to_numeric(numeric_df['cuOpt Vehicles'], errors='coerce')
numeric_df['BK Vehicles']    = pd.to_numeric(numeric_df['BK Vehicles'],    errors='coerce')
numeric_df['cuOpt Cost_num'] = pd.to_numeric(
    numeric_df['cuOpt Cost'].str.replace(',', ''), errors='coerce'
)
numeric_df['BK Cost_num']    = pd.to_numeric(
    numeric_df['BK Cost'].str.replace(',', ''),    errors='coerce'
)
numeric_df['cost_gap_num']   = (
    numeric_df['Cost Gap (%)'].str.rstrip('%').astype(float, errors='ignore')
)

family_stats = (
    numeric_df
    .groupby('Family')
    .agg(
        Instances        = ('cuOpt Vehicles', 'count'),
        Avg_cuOpt_Veh    = ('cuOpt Vehicles', 'mean'),
        Avg_BK_Veh       = ('BK Vehicles',    'mean'),
        Avg_cuOpt_Cost   = ('cuOpt Cost_num',  'mean'),
        Avg_BK_Cost      = ('BK Cost_num',     'mean'),
    )
)
family_stats['Avg_Cost_Gap_%'] = (
    (family_stats['Avg_cuOpt_Cost'] / family_stats['Avg_BK_Cost'] - 1) * 100
).round(2)

print("\nPer-family aggregate statistics:")
try:
    display(family_stats)
except NameError:
    print(family_stats.to_string())

## 9. Conclusion

This notebook benchmarked **NVIDIA cuOpt** (via the OCI AI Accelerator Pack Vehicle Route Optimizer)
across all 60 instances of the Gehring & Homberger 200-customer CVRPTW benchmark.

### Key Observations

- **Clustered instances (C1, C2)**: Customer clusters naturally constrain routes to compact geographic
  zones, typically yielding lower absolute travel costs but requiring careful time-window scheduling.
  cuOpt generally performs well here.

- **Random instances (R1, R2)**: Uniformly scattered customers create longer, more variable inter-stop
  distances. The wider time windows in the R2 family allow the solver more flexibility to consolidate
  routes and reduce vehicle counts.

- **Mixed instances (RC1, RC2)**: The combination of clustered and random customers often produces
  the highest absolute costs. cuOpt still finds good solutions, though the gap to the best-known
  result may be slightly larger than for the C families.

- **Time-window width**: Wide time windows (C2, R2, RC2) generally lead to fewer vehicles and lower
  costs compared to their narrow counterparts (C1, R1, RC1), because the solver has more scheduling
  flexibility to consolidate stops.

- **Solver time limit**: Increasing `TIME_LIMIT` gives cuOpt more opportunity to refine routes and
  can materially reduce the gap to the best-known solution, especially for the harder R1 and RC1
  families. Re-run this notebook with a higher `TIME_LIMIT` to explore the quality–speed trade-off.

## 10. Cleanup
To avoid ongoing GPU/OKE costs after testing:
1. In the OCI Console, locate the Resource Manager stack used to deploy the Vehicle Route Optimizer.
2. Run **Destroy** on the stack (or follow your team’s standard teardown process).
3. Verify that GPU compute resources, load balancers, block volumes, and any related networking components were deleted as expected.